<a href="https://colab.research.google.com/github/oxanaadry-svg/python_sis_26_2/blob/oxana_silva/aula-oxana_silva_aula11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Alteracoes: Cadastro, listagem, busca e calculo de notas fiscais

# Importa as bibliotecas utilizadas no programa
from pathlib import Path
import xml.etree.ElementTree as et
from openpyxl import Workbook


# Cria a pasta onde serao armazenadas as notas fiscais
pasta = Path("notas_xml")
pasta.mkdir(exist_ok=True)


# Funcao responsavel por mostrar o menu
def menu():

    print("\n================================")
    print("   SISTEMA DE NOTAS FISCAIS")
    print("================================")
    print("1 - Cadastrar nota fiscal")
    print("2 - Listar notas fiscais")
    print("3 - Buscar nota fiscal")
    print("4 - Calcular total das notas")
    print("5 - Exportar notas para Excel")
    print("6 - Sair")
    print("================================")


# Funcao para cadastrar uma nota fiscal
def cadastrar_nota():

    print("\n--- CADASTRO DE NOTA FISCAL ---")

    # Solicita os dados da nota
    numero = input("Numero da nota: ")
    data = input("Data da nota (AAAA-MM-DD): ")
    fornecedor = input("Nome do fornecedor: ")
    cnpj = input("CNPJ do fornecedor: ")
    valor = input("Valor total da nota: ")

    # Verifica se o valor informado e numerico
    try:
        valor = float(valor)

        # Verifica se o valor e positivo
        if valor <= 0:
            print("O valor deve ser maior que zero!")
            return

    except ValueError:
        print("Valor invalido! Digite um numero.")
        return

    # Define o nome do arquivo XML
    arquivo = pasta / f"nota_{numero}.xml"

    # Verifica se ja existe uma nota com esse numero
    if arquivo.exists():
        print("Ja existe uma nota com esse numero!")
        return

    # Cria o elemento principal do XML
    raiz = et.Element("nota")

    # Adiciona os dados dentro do XML
    et.SubElement(raiz, "numero").text = numero
    et.SubElement(raiz, "data").text = data
    et.SubElement(raiz, "fornecedor").text = fornecedor
    et.SubElement(raiz, "cnpj").text = cnpj
    et.SubElement(raiz, "valor").text = f"{valor:.2f}"

    # Salva os dados no arquivo XML
    et.ElementTree(raiz).write(
        arquivo,
        encoding="utf-8",
        xml_declaration=True
    )

    print("\nNota fiscal cadastrada com sucesso!")


# Funcao para listar todas as notas fiscais
def listar_notas():

    print("\n--- LISTAGEM DE NOTAS FISCAIS ---")

    # Procura todos os arquivos XML da pasta
    arquivos = sorted(pasta.glob("*.xml"))

    # Verifica se existem notas cadastradas
    if not arquivos:
        print("Nenhuma nota fiscal cadastrada!")
        return

    # Percorre todos os arquivos encontrados
    for arquivo in arquivos:

        try:
            # Abre e interpreta o arquivo XML
            raiz = et.parse(arquivo).getroot()

            # Busca as informacoes dentro do XML
            numero = raiz.findtext("numero")
            data = raiz.findtext("data")
            fornecedor = raiz.findtext("fornecedor")
            cnpj = raiz.findtext("cnpj")
            valor = raiz.findtext("valor")

            # Verifica se todos os campos estao preenchidos
            if not all([numero, data, fornecedor, cnpj, valor]):
                print(f"Arquivo {arquivo.name} possui dados incompletos!")
                continue

            # Mostra os dados da nota fiscal
            print("\n-------------------------------")
            print(f"Numero: {numero}")
            print(f"Data: {data}")
            print(f"Fornecedor: {fornecedor}")
            print(f"CNPJ: {cnpj}")
            print(f"Valor: R$ {float(valor):.2f}")
            print("-------------------------------")

        except (et.ParseError, OSError, ValueError) as erro:

            # Mostra o erro encontrado
            print(f"Erro ao ler {arquivo.name}: {erro}")


# Funcao para buscar uma nota fiscal pelo numero
def buscar_nota():

    print("\n--- BUSCAR NOTA FISCAL ---")

    # Solicita o numero da nota
    numero = input("Digite o numero da nota: ")

    # Localiza o arquivo XML correspondente
    arquivo = pasta / f"nota_{numero}.xml"

    # Verifica se o arquivo existe
    if not arquivo.exists():
        print("Nota fiscal nao encontrada!")
        return

    try:
        # Abre e interpreta o arquivo XML
        raiz = et.parse(arquivo).getroot()

        # Busca as informacoes da nota
        numero = raiz.findtext("numero")
        data = raiz.findtext("data")
        fornecedor = raiz.findtext("fornecedor")
        cnpj = raiz.findtext("cnpj")
        valor = raiz.findtext("valor")

        # Verifica se os campos estao preenchidos
        if not all([numero, data, fornecedor, cnpj, valor]):
            print("A nota possui dados incompletos!")
            return

        # Mostra os dados encontrados
        print("\nNota fiscal encontrada!")
        print("-------------------------------")
        print(f"Numero: {numero}")
        print(f"Data: {data}")
        print(f"Fornecedor: {fornecedor}")
        print(f"CNPJ: {cnpj}")
        print(f"Valor: R$ {float(valor):.2f}")
        print("-------------------------------")

    except (et.ParseError, OSError, ValueError) as erro:

        # Mostra o erro encontrado
        print(f"Erro ao buscar nota: {erro}")


# Funcao para calcular o valor total das notas fiscais
def calcular_total():

    print("\n--- TOTAL DAS NOTAS FISCAIS ---")

    # Cria uma variavel para armazenar a soma
    total = 0.0

    # Cria uma variavel para contar as notas
    quantidade = 0

    # Procura todos os arquivos XML da pasta
    arquivos = sorted(pasta.glob("*.xml"))

    # Verifica se existem notas cadastradas
    if not arquivos:
        print("Nenhuma nota fiscal cadastrada!")
        return

    # Percorre todos os arquivos XML
    for arquivo in arquivos:

        try:
            # Abre e interpreta o arquivo XML
            raiz = et.parse(arquivo).getroot()

            # Busca o valor da nota
            valor = raiz.findtext("valor")

            # Verifica se o valor esta preenchido
            if valor is None or not valor.strip():
                print(f"Valor ausente em {arquivo.name}")
                continue

            # Converte o valor para numero decimal
            valor = float(valor)

            # Soma o valor da nota ao total
            total = total + valor

            # Conta a quantidade de notas
            quantidade = quantidade + 1

        except (et.ParseError, OSError, ValueError) as erro:

            # Mostra o erro encontrado
            print(f"Erro ao calcular {arquivo.name}: {erro}")

    # Mostra o resultado da soma
    print("\nQuantidade de notas:", quantidade)
    print(f"Valor total: R$ {total:.2f}")


# Funcao principal do programa
def main():

    # Define a opcao inicial
    opcao = 0

    # Repete o menu ate o usuario escolher sair
    while opcao != 6:

        # Mostra o menu
        menu()

        # Recebe a opcao escolhida
        try:
            opcao = int(input("Escolha uma opcao: "))

        except ValueError:
            print("Digite somente numeros!")
            continue

        # Verifica a opcao escolhida
        if opcao == 1:
            cadastrar_nota()

        elif opcao == 2:
            listar_notas()

        elif opcao == 3:
            buscar_nota()

        elif opcao == 4:
            calcular_total()

        elif opcao == 5:
            print("Funcao de exportacao sera implementada.")

        elif opcao == 6:
            print("Sistema encerrado!")

        else:
            print("Opcao invalida! Tente novamente.")


# Inicia o programa
main()



   SISTEMA DE NOTAS FISCAIS
1 - Cadastrar nota fiscal
2 - Listar notas fiscais
3 - Buscar nota fiscal
4 - Calcular total das notas
5 - Exportar notas para Excel
6 - Sair
Escolha uma opcao: 67
Opcao invalida! Tente novamente.

   SISTEMA DE NOTAS FISCAIS
1 - Cadastrar nota fiscal
2 - Listar notas fiscais
3 - Buscar nota fiscal
4 - Calcular total das notas
5 - Exportar notas para Excel
6 - Sair
Escolha uma opcao: 3

--- BUSCAR NOTA FISCAL ---
Digite o numero da nota: 98654
Nota fiscal nao encontrada!

   SISTEMA DE NOTAS FISCAIS
1 - Cadastrar nota fiscal
2 - Listar notas fiscais
3 - Buscar nota fiscal
4 - Calcular total das notas
5 - Exportar notas para Excel
6 - Sair
Escolha uma opcao: 4

--- TOTAL DAS NOTAS FISCAIS ---
Nenhuma nota fiscal cadastrada!

   SISTEMA DE NOTAS FISCAIS
1 - Cadastrar nota fiscal
2 - Listar notas fiscais
3 - Buscar nota fiscal
4 - Calcular total das notas
5 - Exportar notas para Excel
6 - Sair
Escolha uma opcao: 6
Sistema encerrado!
